In [1]:
import os

dataset_path = "/Users/justinhoffman/HTR"
images_path = os.path.join(dataset_path, "self_lines")
labels_path = os.path.join(dataset_path, "lines.txt")

samples = []
with open(labels_path, "r") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        parts = line.split()
        filename = parts[0] + ".png"
        text = " ".join(parts[9:]).replace("|", " ")
        image_path = os.path.join(images_path, filename)
        if os.path.exists(image_path):
            samples.append({"image_path": image_path, "text": text})

print(f"Loaded {len(samples)} samples")
print(f"Example: {samples[0]}")

Loaded 336 samples
Example: {'image_path': '/Users/justinhoffman/HTR/self_lines/selfMade_0.png', 'text': 'Machine Learning is an dea to learn from examples'}


In [2]:
import random
random.seed(21)
random.shuffle(samples)

train_size = int(0.70 * len(samples))
val_size = int(0.15 * len(samples))

train_samples = samples[:train_size]
val_samples = samples[train_size:train_size + val_size]
test_samples = samples[train_size + val_size:]

print(f"Train: {len(train_samples)}")
print(f"Val:   {len(val_samples)}")
print(f"Test:  {len(test_samples)}")

Train: 235
Val:   50
Test:  51


In [3]:
import torch
from torch.utils.data import Dataset
from PIL import Image, ImageOps, ImageEnhance
import random

class HandwritingDataset(Dataset):
    def __init__(self, samples, processor, augment=False, max_target_length=128):
        self.samples = samples
        self.processor = processor
        self.augment = augment
        self.max_target_length = max_target_length

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]

        image = Image.open(sample["image_path"]).convert("RGB")
        image = ImageOps.grayscale(image)
        image = ImageOps.autocontrast(image)

        if self.augment:
            angle = random.uniform(-3, 3)
            image = image.rotate(angle, fillcolor=255)
            enhancer = ImageEnhance.Brightness(image)
            image = enhancer.enhance(random.uniform(0.8, 1.2))
            enhancer = ImageEnhance.Contrast(image)
            image = enhancer.enhance(random.uniform(0.8, 1.2))

        image = image.convert("RGB")
        pixel_values = self.processor(images=image, return_tensors="pt").pixel_values.squeeze()

        labels = self.processor.tokenizer(
            sample["text"],
            padding="max_length",
            max_length=self.max_target_length,
            truncation=True,
            return_tensors="pt"
        ).input_ids.squeeze()

        labels[labels == self.processor.tokenizer.pad_token_id] = -100

        return {"pixel_values": pixel_values, "labels": labels}

In [4]:
from torch.utils.data import DataLoader
from transformers import TrOCRProcessor

processor = TrOCRProcessor.from_pretrained("microsoft/trocr-small-handwritten")

train_dataset = HandwritingDataset(train_samples, processor, augment=True)
val_dataset = HandwritingDataset(val_samples, processor, augment=False)
test_dataset = HandwritingDataset(test_samples, processor, augment=False)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches:   {len(val_loader)}")
print(f"Test batches:  {len(test_loader)}")

# Verify one batch
batch = next(iter(train_loader))
print("pixel_values shape:", batch["pixel_values"].shape)
print("labels shape:", batch["labels"].shape)

# Decode a label to verify it matches actual text
sample_labels = batch["labels"][0].clone()
sample_labels[sample_labels == -100] = processor.tokenizer.pad_token_id
decoded = processor.tokenizer.decode(sample_labels, skip_special_tokens=True)
print("Decoded label:", decoded)

Train batches: 59
Val batches:   13
Test batches:  13
pixel_values shape: torch.Size([4, 3, 384, 384])
labels shape: torch.Size([4, 128])
Decoded label: experience, without being explicitly programmed. Instead of


In [5]:
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
from jiwer import cer, wer
import torch

processor = TrOCRProcessor.from_pretrained("microsoft/trocr-small-handwritten")
model = VisionEncoderDecoderModel.from_pretrained("microsoft/trocr-small-handwritten")

model.eval()

predictions = []
ground_truths = []

for sample in test_samples:
    image = Image.open(sample["image_path"]).convert("RGB")
    image = ImageOps.grayscale(image)
    image = ImageOps.autocontrast(image)
    image = image.convert("RGB")

    pixel_values = processor(images=image, return_tensors="pt").pixel_values

    with torch.no_grad():
        generated_ids = model.generate(pixel_values)

    predicted_text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
    predictions.append(predicted_text)
    ground_truths.append(sample["text"])

baseline_cer = cer(ground_truths, predictions)
baseline_wer = wer(ground_truths, predictions)

print(f"Pretrained baseline CER: {baseline_cer:.2%}")
print(f"Pretrained baseline WER: {baseline_wer:.2%}")

Loading weights:   0%|          | 0/360 [00:00<?, ?it/s]

VisionEncoderDecoderModel LOAD REPORT from: microsoft/trocr-small-handwritten
Key                         | Status  | 
----------------------------+---------+-
encoder.pooler.dense.bias   | MISSING | 
encoder.pooler.dense.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/transformers/generation/utils.py:1569: UserWarning: Using the model-agnostic default `max_length` (=21) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


Pretrained baseline CER: 13.69%
Pretrained baseline WER: 48.63%


### Pre Fine-tuning Baseline
CER: 13.69%
WER: 48.63%